In [ ]:
import sys

sys.path.insert(0, "/workspaces/nstu-practice-spring-2026")

import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

from students.naumov.lesson2 import Exercise

# Загружаем датасет
iris = load_iris()
X = iris.data
y = iris.target

# Бинарная задача: setosa (класс 0) vs остальные
y_binary = (y == 0).astype(int)

# Разбиваем на train/val/test
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y_binary, test_size=0.2, random_state=42, stratify=y_binary
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42, stratify=y_train_val
)

# Нормализация по train
mean = X_train.mean(axis=0)
std = X_train.std(axis=0)
X_train_norm = (X_train - mean) / std
X_val_norm = (X_val - mean) / std
X_test_norm = (X_test - mean) / std

print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

In [ ]:
def evaluate_model(lr, batch_size, X_train, X_val, y_train, y_val, n_epochs=25, seed=42):
    model = Exercise.create_logistic_model(num_features=4, rng=np.random.default_rng(seed))
    Exercise.fit(model, x=X_train, y=y_train, lr=lr, n_epoch=n_epochs, batch_size=batch_size)

    y_pred_proba = model.predict(X_val)
    y_pred = (y_pred_proba >= 0.5).astype(int)

    accuracy = np.mean(y_pred == y_val)
    auroc = roc_auc_score(y_val, y_pred_proba)

    return accuracy, auroc, model

In [ ]:
learning_rates = [0.001, 0.003, 0.005, 0.007, 0.009, 0.01, 0.02, 0.03, 0.05, 0.07, 0.1, 0.15, 0.2, 0.3, 0.5, 0.7, 1.0]
batch_sizes = [4, 8, 16, 32, 64, None]
seeds = [1, 2, 3, 4, 5]

results = []

for lr in learning_rates:
    for bs in batch_sizes:
        auroc_scores = []
        accuracy_scores = []

        for seed in seeds:
            accuracy, auroc, _ = evaluate_model(lr, bs, X_train_norm, X_val_norm, y_train, y_val, seed=seed)
            auroc_scores.append(auroc)
            accuracy_scores.append(accuracy)

        results.append(
            {
                "lr": lr,
                "batch_size": bs,
                "mean_auroc": np.mean(auroc_scores),
                "std_auroc": np.std(auroc_scores),
                "mean_accuracy": np.mean(accuracy_scores),
                "std_accuracy": np.std(accuracy_scores),
            }
        )

df = pd.DataFrame(data=results)
df["stability_auroc"] = df["mean_auroc"] - df["std_auroc"]
df["stability_accuracy"] = df["mean_accuracy"] - df["std_accuracy"]

print("ТОП по AUROC:")
print(
    df.sort_values("stability_auroc", ascending=False)[
        ["lr", "batch_size", "mean_auroc", "std_auroc", "stability_auroc", "mean_accuracy"]
    ]
    .head(5)
    .to_string()
)